In [98]:
from langchain_community.document_loaders import PyPDFLoader,TextLoader
from langchain_text_splitters import NLTKTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [93]:
loader = TextLoader(
    "C:/Users/USER/Documents/machine-learning/projects/slgs/rag-poc/app/uploads/from_continuous_deployment_with_jenkins_pipelines_to_mcp_and_llm.md",
    encoding="utf-8"
)
import json

documents = loader.load()




# for d in documents:
#     print(d.page_content)
documents[0].page_content

'From Continuous Deployment with Jenkins Pipelines to MCP and LLM.\n\nmarjune\nFollow\n5 min read\n\nJun 26, 2025\n\nWith my progress in creating my own MCP (Model Context Protocol) server to manage a Kubernetes cluster, my primary goal is to learn how to utilize LLM for tasks beyond code generation and public knowledge questions. As I discussed in my previous article,\n\nHow I Built a Personal Docker-based MCP to manage a K8s cluster.\n\nI created a few simple tools to interface with a Kubernetes cluster. Moving forward, I want to add more capability where, instead of using a CI/CD pipeline to deploy my application, LLM will take care of the deployment on my behalf, based on certain standards, so that LLM will not assume how to proceed. Since LLM can perfectly reason out, I\'ll communicate in a way that makes it seem like my intelligent assistant\nis handling the task, removing the technical overhead associated with operating tools like Jenkins or any other CI/CD.\n\n**Application Dep

In [ ]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_api_key,temperature=0)

chunks = {}

def create_new_chunk(chunk_id, proposition):
    summary_llm = llm.with_structured_output(ChunkMeta)

    summary_prompt_template = ChatPromptTemplate.from_messages([
        (
            "system",
            "Generate a new summary and a title based on the propositions",
        ),
        (
            "user",
            "propositions:{propositions}"
        )
    ])

    summary_chain = summary_prompt_template | summary_llm

    chunk_meta = summary_chain.invoke(
        {
            "propositions": [proposition]
        }
    )

    chunks[chunk_id] = {
        "summary": chunk_meta.summary,
        "title": chunk_meta.title,
        "propositions": [proposition],
    }

    return chunk_id


class ChunkMeta(BaseModel):
    title: str = Field(description="The title of the chunk.")
    summary: str = Field(description="The summary of the chunk.")

def add_proposition(chunk_id, proposition):

    # print(f"CHUNK ID: {chunk_id}")
    # print(f"PROPOSITION: {proposition}")
    summary_llm = llm.with_structured_output(ChunkMeta)

    summary_prompt_template = ChatPromptTemplate.from_messages([
        (
            "system",
            "If the current_summary and title is still valid for the propositions return them."
            "If not generate a new summary and a title based on the propositions."
        ),
        (
            "user",
            "current_summary:{current_summary}\n current_title:{current_title}\n\n propositions:{propositions}",
        )
    ])
    summary_chain = summary_prompt_template | summary_llm

    chunk = chunks[chunk_id]

    current_summary = chunk["summary"]
    current_title = chunk["title"]
    current_propositions = chunk["propositions"]
    all_propositions = current_propositions + [proposition]
    chunk_meta = summary_chain.invoke(
        {
            "current_summary": current_summary,
            "current_title": current_title,
            "propositions": all_propositions
        }
    )

    chunk["summary"] = chunk_meta.summary
    chunk["title"] = chunk_meta.title
    chunk["propositions"] = all_propositions

    return chunk


def use_find_chunk_and_push_proposition(proposition):

    class ChunkID(BaseModel):
        chunk_id: int = Field(description="The chunk id.")

    allocation_llm = llm.with_structured_output(ChunkID)
    allocation_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You assign propositions to semantic chunks.

                    Return the best matching chunk_id.
                    If none match, return a new chunk_id.
                    Return only the chunk_id.
                    """,),
                                (
                    "user",
                    """Proposition:
                    {proposition}

                    Existing chunks:
                    {chunks_summaries}
                """,
            ),
        ]
    )

    allocation_chain = allocation_prompt | allocation_llm
    chunks_summaries = {
        chunk_id: chunk["summary"] for chunk_id, chunk in chunks.items()
    }

    best_chunk_id = allocation_chain.invoke(
        {"proposition": proposition, "chunks_summaries": chunks_summaries}
    ).chunk_id

    if best_chunk_id not in chunks:
        best_chunk_id = create_new_chunk(best_chunk_id, proposition)
        return

    add_proposition(best_chunk_id, proposition)

chunks

{}

In [115]:
for d in documents:
    print(d.page_content)

From Continuous Deployment with Jenkins Pipelines to MCP and LLM.

marjune
Follow
5 min read

Jun 26, 2025

With my progress in creating my own MCP (Model Context Protocol) server to manage a Kubernetes cluster, my primary goal is to learn how to utilize LLM for tasks beyond code generation and public knowledge questions. As I discussed in my previous article,

How I Built a Personal Docker-based MCP to manage a K8s cluster.

I created a few simple tools to interface with a Kubernetes cluster. Moving forward, I want to add more capability where, instead of using a CI/CD pipeline to deploy my application, LLM will take care of the deployment on my behalf, based on certain standards, so that LLM will not assume how to proceed. Since LLM can perfectly reason out, I'll communicate in a way that makes it seem like my intelligent assistant
is handling the task, removing the technical overhead associated with operating tools like Jenkins or any other CI/CD.

**Application Deployment**

I crea

In [ ]:

chunks = {}
for d in documents:
    sentences = sent_tokenize(d.page_content)

    for s in sentences:
        use_find_chunk_and_push_proposition(s)
        print(s)


# print(json_data)
chunks






null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null
null


{1: {'summary': "This piece discusses the author's journey in creating a Model Context Protocol (MCP) server for managing Kubernetes clusters, focusing on the integration of large language models (LLMs) in software development. The author aims to explore LLM applications beyond code generation and public knowledge questions, building on previous work related to Docker-based MCP tools. The article also emphasizes the importance of designing focused and narrowly scoped tools as the MCP development progresses.",
  'title': 'From Continuous Deployment with Jenkins Pipelines to MCP and LLM',
  'propositions': ['From Continuous Deployment with Jenkins Pipelines to MCP and LLM.',
   'marjune\nFollow\n5 min read\n\nJun 26, 2025\n\nWith my progress in creating my own MCP (Model Context Protocol) server to manage a Kubernetes cluster, my primary goal is to learn how to utilize LLM for tasks beyond code generation and public knowledge questions.',
   'As I discussed in my previous article,\n\nHow

In [111]:
# sentences = [
#     "LangChain is a framework for building LLM apps.",
#     "It supports retrieval augmented generation.",
#     "PGVector stores embeddings inside PostgreSQL."
# ]

# for i, sentence in enumerate(sentences):
#    print(f"SENTENCE: {sentence}")
#    print(use_find_chunk_and_push_proposition(sentence))

# print(chunks)

chunks

{1: {'summary': 'This piece explores the evolution of continuous deployment practices, focusing on the shift from traditional Jenkins pipelines to modern methodologies involving Multi-Cloud Platforms (MCP) and Large Language Models (LLM). It highlights the benefits and challenges of adopting these advanced technologies in the deployment process, particularly in the context of developing a Model Context Protocol (MCP) server for managing Kubernetes clusters and leveraging LLM for diverse applications.',
  'title': 'From Continuous Deployment with Jenkins Pipelines to MCP and LLM',
  'propositions': ['From Continuous Deployment with Jenkins Pipelines to MCP and LLM.',
   'marjune\nFollow\n5 min read\n\nJun 26, 2025\n\nWith my progress in creating my own MCP (Model Context Protocol) server to manage a Kubernetes cluster, my primary goal is to learn how to utilize LLM for tasks beyond code generation and public knowledge questions.',
   'As I discussed in my previous article,\n\nHow I Buil

In [61]:

from langchain_aws import ChatBedrock
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_SESSION_TOKEN"] = os.getenv("AWS_SESSION_TOKEN")
os.environ["AWS_DEFAULT_REGION"] = os.getenv("AWS_DEFAULT_REGION")


model = ChatBedrock(
    model_id="arn:aws:bedrock:ap-southeast-1:408897322877:application-inference-profile/pf3b7fwx6on0",
    region_name="ap-southeast-1",
    provider="anthropic",

)


model

Both api_key and AWS credentials were provided. Using api_key for authentication; AWS credentials will be ignored.
Both api_key and AWS credentials were provided. Using api_key for authentication; AWS credentials will be ignored.


ChatBedrock(client=<botocore.client.BedrockRuntime object at 0x000002600DD7C980>, bedrock_client=<botocore.client.Bedrock object at 0x000002600DD7CC20>, region_name='ap-southeast-1', aws_access_key_id=SecretStr('**********'), aws_secret_access_key=SecretStr('**********'), aws_session_token=SecretStr('**********'), bedrock_api_key=SecretStr('**********'), provider='anthropic', model_id='arn:aws:bedrock:ap-southeast-1:408897322877:application-inference-profile/pf3b7fwx6on0', base_model_id='anthropic.claude-opus-4-5-20251101-v1:0', model_kwargs={}, profile={})

In [66]:
from langchain_core.messages import HumanMessage

def _invoke_bedrock_image(image_b64, mime_type):
    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": (
                    "You are an expert AI assistant, you are tasked with extracting the entire text from any PDF document. The document can be simple, complex, or even scanned, this shouldn't matter to you."
                    "You will be given the entire PDF as input. Start examining the document page by page, when you come across text, extract it as is don't convert it into another format like HTML or Markdown. If you come across images, replace them with a very detailed description of the image while taking into consideration the context around it."
                    "When you come across tables, describe them too like the image. The description should be very detailed and in a way that someone will understand the table without seeing it."
                    "Make sure to keep the structure of the document, if there are sections, subsections, bullet points, or numbered lists, make sure to keep them as is. If there are any headers, footers, page numbers, remove them."
                    "The final output should be a clean, well-structured text that represents the content of the entire PDF document as closely as possible to how a human would see it with their eyes when reading the document. Don't say anything else, just output the text you extracted from the PDF."
                    "Here is the PDF:"
                ),
            },
            {
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": mime_type,
                    "data": image_b64,
                },
            },
        ]
    )

    return model.invoke([message]) 

In [67]:
import base64
def _encode_image(self, image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode()

In [68]:
import io
import base64
import mimetypes
from pdf2image import convert_from_path

def parse_document(image_path):

    print(f"IMAGE PATH: {image_path}")


    if not image_path:
        return None
    
    if not os.path.exists(image_path):
        raise ValueError(f"Image not found: {image_path}")
    
    mime_type, _ = mimetypes.guess_type(image_path)
    results = []

    if mime_type in ["image/png", "image/jpeg", "image/webp"]:
        image_b64 = _encode_image(image_path)
        response = _invoke_bedrock_image(image_b64, mime_type)
        results.append(response.content)

    elif mime_type == "application/pdf":
        pages = convert_from_path(
            image_path,
            dpi=200,
            fmt="png",
            poppler_path=r"C:\poppler-25.12.0\Library\bin"
        )

        for i, page in enumerate(pages):
            buffer = io.BytesIO()
            page.save(buffer, format="PNG")


            image_b64 = base64.b64encode(buffer.getvalue()).decode()

            response = _invoke_bedrock_image(image_b64,"image/png")

            print(f"RESPONSE: {response.content}")

            results.append(response.content)
    else:
        raise ValueError(f"Unsupported file type: {mime_type}")
    
    return results

In [69]:
parse_document("C:/Users/USER/Documents/machine-learning/projects/slgs/rag-poc/app/uploads/from_continuous_deployment_with_jenkins_pipelines_to_mcp_and_llm_497953d962e7.pdf")

IMAGE PATH: C:/Users/USER/Documents/machine-learning/projects/slgs/rag-poc/app/uploads/from_continuous_deployment_with_jenkins_pipelines_to_mcp_and_llm_497953d962e7.pdf
RESPONSE: From Continuous Deployment with Jenkins Pipelines to MCP and LLM.

marjune
Follow
5 min read

Jun 26, 2025

With my progress in creating my own MCP (Model Context Protocol) server to manage a Kubernetes cluster, my primary goal is to learn how to utilize LLM for tasks beyond code generation and public knowledge questions. As I discussed in my previous article,

How I Built a Personal Docker-based MCP to manage a K8s cluster.

I created a few simple tools to interface with a Kubernetes cluster. Moving forward, I want to add more capability where, instead of using a CI/CD pipeline to deploy my application, LLM will take care of the deployment on my behalf, based on certain standards, so that LLM will not assume how to proceed. Since LLM can perfectly reason out, I'll communicate in a way that makes it seem like 

["From Continuous Deployment with Jenkins Pipelines to MCP and LLM.\n\nmarjune\nFollow\n5 min read\n\nJun 26, 2025\n\nWith my progress in creating my own MCP (Model Context Protocol) server to manage a Kubernetes cluster, my primary goal is to learn how to utilize LLM for tasks beyond code generation and public knowledge questions. As I discussed in my previous article,\n\nHow I Built a Personal Docker-based MCP to manage a K8s cluster.\n\nI created a few simple tools to interface with a Kubernetes cluster. Moving forward, I want to add more capability where, instead of using a CI/CD pipeline to deploy my application, LLM will take care of the deployment on my behalf, based on certain standards, so that LLM will not assume how to proceed. Since LLM can perfectly reason out, I'll communicate in a way that makes it seem like my intelligent assistant",
 'is handling the task, removing the technical overhead associated with operating tools like Jenkins or any other CI/CD.\n\n**Application 